## Download ACS 5-Year Data for Virginia Census Tracts

Downloads 2018 ACS 5-Year estimates at the census tract level for Virginia.

Variables downloaded:
- Population
- Median household income
- Housing units
- Race
- Educational attainment

In [10]:
import requests
import pandas as pd
from pathlib import Path

DATA_DIR = next(
    p / "data" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists()
)

API_KEY = "a8babf64d8f6e73e016a236b3c0cff22f133340c"  
BASE_URL = "https://api.census.gov/data/2018/acs/acs5"
STATE = "51"  # Virginia FIPS code

In [7]:
# Variable definitions
# The Census API has a 50-variable limit per request, so we split into groups

variables = {
    # Population
    "B01001_001E": "total_population",

    # Median household income
    "B19013_001E": "median_household_income",

    # Housing units
    "B25001_001E": "total_housing_units",

    # Race (B02001)
    "B02001_001E": "race_total",
    "B02001_002E": "race_white_alone",

    # Educational attainment 25+ (B15003)
    "B15003_001E": "edu_total_25plus",
    "B15003_002E": "edu_no_schooling",
    "B15003_017E": "edu_high_school_diploma",
    "B15003_018E": "edu_ged",
    "B15003_019E": "edu_some_college_less_1yr",
    "B15003_020E": "edu_some_college_1yr_plus",
    "B15003_021E": "edu_associates",
    "B15003_022E": "edu_bachelors",
    "B15003_023E": "edu_masters",
    "B15003_024E": "edu_professional",
    "B15003_025E": "edu_doctorate",
}

print(f"Total variables to download: {len(variables)}")

Total variables to download: 16


In [8]:
def fetch_acs(variable_codes, state, api_key):
    """
    Fetch ACS 5-year variables for all census tracts in a state.
    Returns a DataFrame.
    """
    # Always include NAME for tract identification
    get_vars = "NAME," + ",".join(variable_codes)

    params = {
        "get": get_vars,
        "for": "tract:*",
        "in": f"state:{state}",
        "key": api_key,
    }

    r = requests.get(BASE_URL, params=params)
    if r.status_code != 200:
        print(f"Error {r.status_code}: {r.text[:300]}")
        return pd.DataFrame()

    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

In [11]:
# Fetch all variables (split into batches of 45 to stay under the 50-variable limit)
var_codes = list(variables.keys())
batch_size = 45
batches = [var_codes[i:i + batch_size] for i in range(0, len(var_codes), batch_size)]

dfs = []
for i, batch in enumerate(batches):
    print(f"Fetching batch {i+1}/{len(batches)}...")
    df = fetch_acs(batch, STATE, API_KEY)
    dfs.append(df)

# Merge all batches on state, county, tract
result = dfs[0]
for df in dfs[1:]:
    result = result.merge(df, on=["NAME", "state", "county", "tract"], how="outer")

print(f"\nShape: {result.shape}")
result.head()

Fetching batch 1/1...

Shape: (1907, 20)


,NAME,B01001_001E,B19013_001E,B25001_001E,B02001_001E,B02001_002E,B15003_001E,B15003_002E,B15003_017E,B15003_018E,B15003_019E,B15003_020E,B15003_021E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,state,county,tract
0,"Census Tract 44, Norfolk city, Virginia",1537,51071,596,1537,122,1065,10,178,27,93,305,57,171,87,0,8,51,710,004400
1,"Census Tract 9503, Nelson County, Virginia",4619,59500,2559,4619,3571,3395,81,980,217,192,401,151,639,300,23,22,51,125,950300
2,"Census Tract 9508, Essex County, Virginia",3757,65804,1992,3757,2408,2474,0,873,100,225,521,156,159,37,0,0,51,057,950800
3,"Census Tract 9506, Essex County, Virginia",3612,53953,1761,3612,1842,2670,61,902,112,143,412,110,311,100,8,25,51,057,950600
4,"Census Tract 9507, Essex County, Virginia",3667,37500,2094,3667,1879,2671,33,761,197,165,375,126,471,137,28,15,51,057,950700


In [12]:
# Rename columns and create GEOID for joining to census shapefile
result = result.rename(columns=variables)
result["GEOID"] = result["state"] + result["county"] + result["tract"]

# Convert numeric columns from string to numeric
numeric_cols = list(variables.values())
result[numeric_cols] = result[numeric_cols].apply(pd.to_numeric, errors="coerce")

# Calculate percentage of white residents
result["pct_white"] = result["race_white_alone"] / result["race_total"] * 100

# Calculate percentage of adults with at least a high school diploma
edu_hs_or_above = (
    result["edu_high_school_diploma"] +
    result["edu_ged"] +
    result["edu_some_college_less_1yr"] +
    result["edu_some_college_1yr_plus"] +
    result["edu_associates"] +
    result["edu_bachelors"] +
    result["edu_masters"] +
    result["edu_professional"] +
    result["edu_doctorate"]
)
result["pct_hs_or_above"] = edu_hs_or_above / result["edu_total_25plus"] * 100

# Reorder columns — put calculated percentages after GEOID/NAME
result = result[["GEOID", "NAME", "state", "county", "tract", "pct_white", "pct_hs_or_above"] + numeric_cols]

print(f"Tracts: {len(result):,}")
print(f"Missing median income: {result['median_household_income'].isna().sum()}")
result.head()

Tracts: 1,907
Missing median income: 0


,GEOID,NAME,state,county,tract,pct_white,pct_hs_or_above,total_population,median_household_income,total_housing_units,...,edu_no_schooling,edu_high_school_diploma,edu_ged,edu_some_college_less_1yr,edu_some_college_1yr_plus,edu_associates,edu_bachelors,edu_masters,edu_professional,edu_doctorate
0,51710004400,"Census Tract 44, Norfolk city, Virginia",51,710,004400,7.937541,86.948357,1537,51071,596,...,10,178,27,93,305,57,171,87,0,8
1,51125950300,"Census Tract 9503, Nelson County, Virginia",51,125,950300,77.311106,86.156112,4619,59500,2559,...,81,980,217,192,401,151,639,300,23,22
2,51057950800,"Census Tract 9508, Essex County, Virginia",51,057,950800,64.093692,83.710590,3757,65804,1992,...,0,873,100,225,521,156,159,37,0,0
3,51057950600,"Census Tract 9506, Essex County, Virginia",51,057,950600,50.996678,79.513109,3612,53953,1761,...,61,902,112,143,412,110,311,100,8,25
4,51057950700,"Census Tract 9507, Essex County, Virginia",51,057,950700,51.240796,85.174092,3667,37500,2094,...,33,761,197,165,375,126,471,137,28,15


In [ ]:
#Filter only relevant columns
result_filtered = result[[
    "GEOID", "NAME", "state", "county", "tract",                                                                                                                                         
    "total_population", "median_household_income", "total_housing_units",                                                                                                                
    "pct_white", "pct_hs_or_above"                                                                                                                                                       
]]                                                                                                                                                                                       
                
print(result_filtered.shape)                                                                                                                                                                 
result_filtered.head()

(1907, 10)


,GEOID,NAME,state,county,tract,total_population,median_household_income,total_housing_units,pct_white,pct_hs_or_above
0,51710004400,"Census Tract 44, Norfolk city, Virginia",51,710,004400,1537,51071,596,7.937541,86.948357
1,51125950300,"Census Tract 9503, Nelson County, Virginia",51,125,950300,4619,59500,2559,77.311106,86.156112
2,51057950800,"Census Tract 9508, Essex County, Virginia",51,057,950800,3757,65804,1992,64.093692,83.710590
3,51057950600,"Census Tract 9506, Essex County, Virginia",51,057,950600,3612,53953,1761,50.996678,79.513109
4,51057950700,"Census Tract 9507, Essex County, Virginia",51,057,950700,3667,37500,2094,51.240796,85.174092


In [14]:
# Save
out_path = DATA_DIR / "processed" / "usa_data" / "va_acs_2018_tract.csv"
result_filtered.to_csv(out_path, index=False)
print(f"Saved {len(result):,} rows to va_acs_2018_tract.csv")

Saved 1,907 rows to va_acs_2018_tract.csv
